# NB-02_controles_y_ajustes_iniciales

Process Flow SAS: **Controles y ajustes iniciales** — `PFD-I32F27oZ8ICuby6y`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

ANIO = None  # &ANIO — nadie declaró su valor
TRIM = None  # &TRIM — nadie declaró su valor
anio = None  # &anio — nadie declaró su valor

faltantes = [n for n, v in {"ANIO": ANIO, "TRIM": TRIM, "anio": anio}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
import bcchapi
from datetime import date
from sqlalchemy import text

# Conexión a BD — editable acá; SASMIG_DB_URL (orquestador) tiene
# prioridad si está definida (SUPUESTO: verificar servidor y base
# antes de correr contra datos reales).
config_db = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=PLATDAT,1433;"
    "DATABASE=GOBGENER;"
    "Authentication=ActiveDirectoryIntegrated;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "MARS_Connection=Yes;"
)
engine = sqlalchemy.create_engine(
    os.environ.get("SASMIG_DB_URL", f"mssql+pyodbc:///?odbc_connect={config_db}"),
    pool_pre_ping=True,
    fast_executemany=True,
)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-02_controles_y_ajustes_iniciales.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## S1

Construye y actualiza las tablas base de ajuste (SIFMI, dividendos de hogares, CNT desde el Banco Central, utilidades reinvertidas, ajustes varios y FBCF del sector financiero) que alimentan la síntesis trimestral CNSI; el tramo final que agrega bonos emitidos en el exterior queda pendiente por depender de un archivo del servidor SAS sin ruta migrada / Ajusta signo y reclasifica bonos del RM activo (CA 36→51021/AF.42) y elimina el bono de sector 6 CA 36 desde la CI, acumulando ambos ajustes en la tabla de ajustes varios

*confianza: low · verificador: approve · SAS: PROC IMPORT (Excel) + PROC SQL UNION ALL/CREATE/DELETE/UPDATE + PROC HTTP/LIBNAME JSON (API BDE) + PROC DATASETS APPEND + PROC SQL UPDATE + CREATE TABLE con agregación y GROUP BY + DATA step SET acumulativo + DROP TABLE*

In [ ]:
# ========= S1 =========
# IMPORTA DATA
sifmi = pd.read_excel(Path("data") / "CONTROLES" / "SIFMI.xlsx", sheet_name="SIFMI_SAS")
_log("sifmi", sifmi)


In [ ]:
# CREA BASE DE DATOS SIFMI. CIERRE 2021: INCORPORA SECTORES GOB, SEGUROS Y AUXILIARES
hoy = pd.Timestamp(date.today())

def _bloque_sifmi(sector, c_cagente, c_entrada, dato):
    return pd.DataFrame({
        "MONEDA": "P",
        "AÑO": sifmi["Año"],
        "TRIM": sifmi["Trimestre"],
        "SECTOR": sector,
        "C_CUENTA": "YG",
        "C_CAGENTE": c_cagente,
        "C_ENTRADA": c_entrada,
        "DATO": dato,
        "C_SCN": "D.41",
        "N_SCN": "Intereses",
        "FUENTE": "DI_Aj_SIFMI",
        "FECHA": hoy,
    })

bloques = [
    _bloque_sifmi(51, "321", "D", sifmi["Empresas_pagados"] * -1),
    _bloque_sifmi(511, "321", "D", sifmi["Hogares_pagados"] * -1),
    _bloque_sifmi(41, "321", "D", sifmi["Gob_pagados"] * -1),
    _bloque_sifmi(35, "321", "D", sifmi["Seg_pagados"] * -1),
    _bloque_sifmi(36, "321", "D", sifmi["Aux_pagados"] * -1),
    _bloque_sifmi(321, "53", "H", (sifmi["Hogares_pagados"] + sifmi["Empresas_pagados"] + sifmi["Gob_pagados"] + sifmi["Seg_pagados"] + sifmi["Aux_pagados"]) * -1),
    _bloque_sifmi(51, "321", "H", sifmi["Empresas_recibidos"]),
    _bloque_sifmi(511, "321", "H", sifmi["Hogares_recibidos"]),
    _bloque_sifmi(41, "321", "H", sifmi["Gob_recibidos"]),
    _bloque_sifmi(35, "321", "H", sifmi["Seg_recibidos"]),
    _bloque_sifmi(36, "321", "H", sifmi["Aux_recibidos"]),
    _bloque_sifmi(321, "53", "D", sifmi["Hogares_recibidos"] + sifmi["Empresas_recibidos"] + sifmi["Gob_recibidos"] + sifmi["Seg_recibidos"] + sifmi["Aux_recibidos"]),
]
tablas_sifmi = pd.concat(bloques, ignore_index=True)
cols_sifmi = ["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CUENTA", "C_CAGENTE", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "FECHA"]
tablas_sifmi = tablas_sifmi[cols_sifmi]
# TABLAS.SIFMI: SAS reemplazaba la tabla completa (CREATE TABLE)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.SIFMI"))
    _log("DELETE TABLAS.dbo.SIFMI", res.rowcount)
tablas_sifmi.to_sql("SIFMI", engine, schema="dbo", if_exists="append", index=False)
_log("tablas_sifmi", tablas_sifmi)


In [ ]:
# elimina filas sin año (missing numérico de SAS = NULL en SQL Server)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.SIFMI WHERE AÑO IS NULL"))
    _log("DELETE TABLAS.dbo.SIFMI (año nulo)", res.rowcount)


In [ ]:
# CALCULA PROMEDIO DEL TRIMESTRE A TRABAJAR. AJUSTE DE INICIO DEL PERIODO EN EL PROCESO DE SÍNTESIS PARA EL PERIODO DE COYUNTURA
sql_rp_hh_sum = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_CAGENTE, C_ENTRADA, C_SCN, N_SCN, FUENTE, SUM(DATO) AS DATO
FROM TABLAS.dbo.RP_HH
WHERE AÑO >= 2008 AND TRIM = :trim
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_CAGENTE, C_ENTRADA, C_SCN, N_SCN, FUENTE
"""
rp_hh_sum = pd.read_sql(text(sql_rp_hh_sum), engine, params={"trim": TRIM})
_log("rp_hh_sum", rp_hh_sum)


In [ ]:
# PARA ELIMINAR PERIODO DE COYUNTURA EN CASO QUE SE CORRA ESTE PROG VARIAS VECES
rp_hh_sum = rp_hh_sum[~((rp_hh_sum["AÑO"] == ANIO) & (rp_hh_sum["TRIM"] == TRIM))].copy()
_log("rp_hh_sum", rp_hh_sum)


In [ ]:
# promedio del trimestre de coyuntura para el año a trabajar (CALCULATED 'AÑO'n en el GROUP BY del SAS)
rp_hh_av = (
    rp_hh_sum.assign(**{"AÑO": ANIO})
    .groupby(["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CUENTA", "C_CAGENTE", "C_ENTRADA", "C_SCN", "N_SCN", "FUENTE"], as_index=False)["DATO"]
    .mean()
)
rp_hh_av["FECHA"] = hoy
rp_hh_av["PROC"] = "P"
_log("rp_hh_av", rp_hh_av)


In [ ]:
# ELIMINA DATOS DE COYUNTURA EN TABLA PRINCIPAL (anti-join equivalente al EXISTS del SAS)
claves_coyuntura = rp_hh_av[["AÑO", "TRIM", "PROC"]].drop_duplicates()
work_ds = claves_coyuntura.rename(columns={"AÑO": "anio_c", "TRIM": "trim_c", "PROC": "proc_c"})
work_ds.to_sql("#tmp_rp_hh_av_claves", work_conn, if_exists="replace", index=False)
res = work_conn.execute(text(
    "DELETE t FROM TABLAS.dbo.RP_HH t "
    "WHERE EXISTS (SELECT 1 FROM #tmp_rp_hh_av_claves c "
    "WHERE c.anio_c = t.AÑO AND c.trim_c = t.TRIM AND c.proc_c = t.PROC)"
))
_log("DELETE TABLAS.dbo.RP_HH (coyuntura previa)", res.rowcount)


In [ ]:
# ANEXA PROMEDIO DIVIDENDOS A BASE RP_HH (PROC APPEND FORCE: alinea por nombre de columna)
cols_rp_hh = ["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CUENTA", "C_CAGENTE", "C_ENTRADA", "C_SCN", "N_SCN", "FUENTE", "DATO", "FECHA", "PROC"]
rp_hh_av[cols_rp_hh].to_sql("RP_HH", engine, schema="dbo", if_exists="append", index=False)
_log("rp_hh_av", rp_hh_av)


In [ ]:
# PIB a precios corrientes — API BDE vía SDK oficial (M-006: credenciales por variable de entorno)
client = bcchapi.Siete(os.environ["BDE_USER"], os.environ["BDE_PASS"])
serie_pib = client.cuadromacro(series=["F032.PIB.FLU.N.CLP.EP18.Z.Z.0.T"])
serie_pib = serie_pib.reset_index().rename(columns={"index": "indexDateString"})
serie_pib.columns = ["indexDateString", "value"]
pib = pd.DataFrame({
    "AÑO": pd.to_numeric(serie_pib["indexDateString"].astype(str).str[:4], errors="coerce"),
    "TRIM": pd.to_numeric(serie_pib["indexDateString"].astype(str).str[5:7], errors="coerce"),
    "DATO": pd.to_numeric(serie_pib["value"], errors="coerce") * 1000,
})
pib["FECHA"] = hoy
# WHERE INPUT(...) truthy en SAS: descarta año no numérico/faltante
pib = pib[pib["AÑO"].notna()]
# TABLAS.PIB: SAS reemplazaba la tabla completa (CREATE TABLE)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.PIB"))
    _log("DELETE TABLAS.dbo.PIB", res.rowcount)
pib.to_sql("PIB", engine, schema="dbo", if_exists="append", index=False)
_log("pib", pib)


In [ ]:
# recodifica mes de la serie a número de trimestre (4->2, 7->3, 10->4)
with engine.begin() as conn:
    r1 = conn.execute(text("UPDATE TABLAS.dbo.PIB SET TRIM = 2 WHERE TRIM = 4"))
    r2 = conn.execute(text("UPDATE TABLAS.dbo.PIB SET TRIM = 3 WHERE TRIM = 7"))
    r3 = conn.execute(text("UPDATE TABLAS.dbo.PIB SET TRIM = 4 WHERE TRIM = 10"))
_log("UPDATE TABLAS.dbo.PIB (recodifica trim)", r1.rowcount + r2.rowcount + r3.rowcount)


In [ ]:
# Ingreso de los factores recibidos del RM
serie_irm = client.cuadromacro(series=["F033.IRM.FLU.N.CLP.EP18.0.T"]).reset_index()
serie_irm.columns = ["indexDateString", "value"]
serie_1 = pd.DataFrame({
    "AÑO": pd.to_numeric(serie_irm["indexDateString"].astype(str).str[:4], errors="coerce"),
    "TRIM": pd.to_numeric(serie_irm["indexDateString"].astype(str).str[5:7], errors="coerce"),
})
serie_1 = serie_1[serie_1["AÑO"] >= 2003].copy()
serie_1["SECTOR"] = 6
serie_1["C_CUENTA"] = "YG"
serie_1["C_ENTRADA"] = "D"
serie_1["DATO"] = pd.to_numeric(serie_irm.loc[serie_1.index, "value"], errors="coerce") * 1000
serie_1["C_SCN"] = "D.4"
serie_1["N_SCN"] = "Ingreso de factores recibidos del RM"
serie_1["FUENTE"] = "CNT"
serie_1["FECHA"] = hoy
_log("serie_1", serie_1)


In [ ]:
# Ingreso de los factores pagados al RM
serie_irmp = client.cuadromacro(series=["F033.IRMP.FLU.N.CLP.EP18.0.T"]).reset_index()
serie_irmp.columns = ["indexDateString", "value"]
serie_2 = pd.DataFrame({
    "AÑO": pd.to_numeric(serie_irmp["indexDateString"].astype(str).str[:4], errors="coerce"),
    "TRIM": pd.to_numeric(serie_irmp["indexDateString"].astype(str).str[5:7], errors="coerce"),
})
serie_2 = serie_2[serie_2["AÑO"] >= 2003].copy()
serie_2["SECTOR"] = 6
serie_2["C_CUENTA"] = "YG"
serie_2["C_ENTRADA"] = "H"
serie_2["DATO"] = pd.to_numeric(serie_irmp.loc[serie_2.index, "value"], errors="coerce") * 1000
serie_2["C_SCN"] = "D.4"
serie_2["N_SCN"] = "Ingreso de factores pagados al RM"
serie_2["FUENTE"] = "CNT"
serie_2["FECHA"] = hoy
_log("serie_2", serie_2)


In [ ]:
# Transferencias corrientes recibidas del exterior
serie_tce = client.cuadromacro(series=["F033.TCE.FLU.N.CLP.EP18.0.T"]).reset_index()
serie_tce.columns = ["indexDateString", "value"]
serie_3 = pd.DataFrame({
    "AÑO": pd.to_numeric(serie_tce["indexDateString"].astype(str).str[:4], errors="coerce"),
    "TRIM": pd.to_numeric(serie_tce["indexDateString"].astype(str).str[5:7], errors="coerce"),
})
serie_3 = serie_3[serie_3["AÑO"] >= 2003].copy()
serie_3["SECTOR"] = 6
serie_3["C_CUENTA"] = "YG"
serie_3["C_ENTRADA"] = "D"
serie_3["DATO"] = pd.to_numeric(serie_tce.loc[serie_3.index, "value"], errors="coerce") * 1000
serie_3["C_SCN"] = "D.7"
serie_3["N_SCN"] = "Transferencias corrientes recibidos del RM"
serie_3["FUENTE"] = "CNT"
serie_3["FECHA"] = hoy
_log("serie_3", serie_3)


In [ ]:
# Transferencias corrientes pagadas al exterior
serie_tcep = client.cuadromacro(series=["F033.TCEP.FLU.N.CLP.EP18.0.T"]).reset_index()
serie_tcep.columns = ["indexDateString", "value"]
serie_4 = pd.DataFrame({
    "AÑO": pd.to_numeric(serie_tcep["indexDateString"].astype(str).str[:4], errors="coerce"),
    "TRIM": pd.to_numeric(serie_tcep["indexDateString"].astype(str).str[5:7], errors="coerce"),
})
serie_4 = serie_4[serie_4["AÑO"] >= 2003].copy()
serie_4["SECTOR"] = 6
serie_4["C_CUENTA"] = "YG"
serie_4["C_ENTRADA"] = "H"
serie_4["DATO"] = pd.to_numeric(serie_tcep.loc[serie_4.index, "value"], errors="coerce") * 1000
serie_4["C_SCN"] = "D.7"
serie_4["N_SCN"] = "Transferencias corrientes pagados al RM"
serie_4["FUENTE"] = "CNT"
serie_4["FECHA"] = hoy
_log("serie_4", serie_4)


In [ ]:
# Ahorro externo
serie_aex = client.cuadromacro(series=["F033.AEX.FLU.N.CLP.EP18.0.T"]).reset_index()
serie_aex.columns = ["indexDateString", "value"]
serie_5 = pd.DataFrame({
    "AÑO": pd.to_numeric(serie_aex["indexDateString"].astype(str).str[:4], errors="coerce"),
    "TRIM": pd.to_numeric(serie_aex["indexDateString"].astype(str).str[5:7], errors="coerce"),
})
serie_5 = serie_5[serie_5["AÑO"] >= 2003].copy()
serie_5["SECTOR"] = 6
serie_5["C_CUENTA"] = "YG"
serie_5["C_ENTRADA"] = "D"
serie_5["DATO"] = pd.to_numeric(serie_aex.loc[serie_5.index, "value"], errors="coerce") * 1000
serie_5["C_SCN"] = "B.8"
serie_5["N_SCN"] = "Ahorro externo"
serie_5["FUENTE"] = "CNT"
serie_5["FECHA"] = hoy
_log("serie_5", serie_5)


In [ ]:
# Formación bruta de capital fijo
serie_fkf = client.cuadromacro(series=["F033.FKF.FLU.N.CLP.EP18.0.T"]).reset_index()
serie_fkf.columns = ["indexDateString", "value"]
serie_6 = pd.DataFrame({
    "AÑO": pd.to_numeric(serie_fkf["indexDateString"].astype(str).str[:4], errors="coerce"),
    "TRIM": pd.to_numeric(serie_fkf["indexDateString"].astype(str).str[5:7], errors="coerce"),
})
serie_6 = serie_6[serie_6["AÑO"] >= 2003].copy()
serie_6["SECTOR"] = 53
serie_6["C_CUENTA"] = "Capital"
serie_6["C_ENTRADA"] = "D"
serie_6["DATO"] = pd.to_numeric(serie_fkf.loc[serie_6.index, "value"], errors="coerce") * 1000
serie_6["C_SCN"] = "P.51"
serie_6["N_SCN"] = "Formación bruta de capital fijo"
serie_6["FUENTE"] = "CNT"
serie_6["FECHA"] = hoy
_log("serie_6", serie_6)


In [ ]:
# Variación de existencias
serie_vax = client.cuadromacro(series=["F033.VAX.FLU.N.CLP.EP18.0.T"]).reset_index()
serie_vax.columns = ["indexDateString", "value"]
serie_7 = pd.DataFrame({
    "AÑO": pd.to_numeric(serie_vax["indexDateString"].astype(str).str[:4], errors="coerce"),
    "TRIM": pd.to_numeric(serie_vax["indexDateString"].astype(str).str[5:7], errors="coerce"),
})
serie_7 = serie_7[serie_7["AÑO"] >= 2003].copy()
serie_7["SECTOR"] = 53
serie_7["C_CUENTA"] = "Capital"
serie_7["C_ENTRADA"] = "D"
serie_7["DATO"] = pd.to_numeric(serie_vax.loc[serie_7.index, "value"], errors="coerce") * 1000
serie_7["C_SCN"] = "P.52"
serie_7["N_SCN"] = "Variación de existencias"
serie_7["FUENTE"] = "CNT"
serie_7["FECHA"] = hoy
_log("serie_7", serie_7)


In [ ]:
# Importación de bienes y servicios (el nombre de la columna N_SCN del SAS dice "Importación" aunque la serie es XBS/exportaciones — se copia tal cual el original)
serie_xbs = client.cuadromacro(series=["F033.XBS.FLU.N.CLP.EP18.0.T"]).reset_index()
serie_xbs.columns = ["indexDateString", "value"]
serie_8 = pd.DataFrame({
    "AÑO": pd.to_numeric(serie_xbs["indexDateString"].astype(str).str[:4], errors="coerce"),
    "TRIM": pd.to_numeric(serie_xbs["indexDateString"].astype(str).str[5:7], errors="coerce"),
})
serie_8 = serie_8[serie_8["AÑO"] >= 2003].copy()
serie_8["SECTOR"] = 6
serie_8["C_CUENTA"] = "Producción"
serie_8["C_ENTRADA"] = "D"
serie_8["DATO"] = pd.to_numeric(serie_xbs.loc[serie_8.index, "value"], errors="coerce") * 1000
serie_8["C_SCN"] = "P.7"
serie_8["N_SCN"] = "Importación de bienes y servicios"
serie_8["FUENTE"] = "CNT"
serie_8["FECHA"] = hoy
_log("serie_8", serie_8)


In [ ]:
# Exportación de bienes y servicios (el nombre de la columna N_SCN del SAS dice "Exportación" aunque la serie es IBS/importaciones — se copia tal cual el original)
serie_ibs = client.cuadromacro(series=["F033.IBS.FLU.N.CLP.EP18.0.T"]).reset_index()
serie_ibs.columns = ["indexDateString", "value"]
serie_9 = pd.DataFrame({
    "AÑO": pd.to_numeric(serie_ibs["indexDateString"].astype(str).str[:4], errors="coerce"),
    "TRIM": pd.to_numeric(serie_ibs["indexDateString"].astype(str).str[5:7], errors="coerce"),
})
serie_9 = serie_9[serie_9["AÑO"] >= 2003].copy()
serie_9["SECTOR"] = 6
serie_9["C_CUENTA"] = "Producción"
serie_9["C_ENTRADA"] = "H"
serie_9["DATO"] = pd.to_numeric(serie_ibs.loc[serie_9.index, "value"], errors="coerce") * 1000
serie_9["C_SCN"] = "P.6"
serie_9["N_SCN"] = "Exportación de bienes y servicios"
serie_9["FUENTE"] = "CNT"
serie_9["FECHA"] = hoy
_log("serie_9", serie_9)


In [ ]:
# une base de datos CNT
cols_cnt = ["AÑO", "TRIM", "SECTOR", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "FECHA"]
cnt = pd.concat([serie_1, serie_2, serie_3, serie_4, serie_5, serie_6, serie_7, serie_8, serie_9], ignore_index=True)[cols_cnt]
# TABLAS.CNT: SAS reemplazaba la tabla completa (DATA step sobre tabla ya existente)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.CNT"))
    _log("DELETE TABLAS.dbo.CNT", res.rowcount)
cnt.to_sql("CNT", engine, schema="dbo", if_exists="append", index=False)
_log("cnt", cnt)


In [ ]:
# recodifica mes de la serie a número de trimestre (4->2, 7->3, 10->4)
with engine.begin() as conn:
    r1 = conn.execute(text("UPDATE TABLAS.dbo.CNT SET TRIM = 2 WHERE TRIM = 4"))
    r2 = conn.execute(text("UPDATE TABLAS.dbo.CNT SET TRIM = 3 WHERE TRIM = 7"))
    r3 = conn.execute(text("UPDATE TABLAS.dbo.CNT SET TRIM = 4 WHERE TRIM = 10"))
_log("UPDATE TABLAS.dbo.CNT (recodifica trim)", r1.rowcount + r2.rowcount + r3.rowcount)


In [ ]:
# las tablas serie_1..serie_9 eran WORK intermedias; en Python no hay tabla física que dropear, quedan liberadas al terminar el nodo


In [ ]:
# IMPORTA DATA FINAL DE UTILIDADES REINVERTIDAS PAGADAS POR EL SECTOR, NUEVO CALCULO TRIMESTRAL CR18. ADEMÁS INCORPORA APERTURA ENTRE BANCOS Y SEGUROS
# DATAROW=2 -> header en la fila 1, datos desde la fila 2
ur_sf_cr18 = pd.read_excel(Path("data") / "INFO_AUX" / "UT_REINVERTIDAS_CR18.xlsx", sheet_name="UR_SF")
# TABLAS.UR_SF_CR18: SAS reemplazaba la tabla completa (PROC IMPORT replace)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.UR_SF_CR18"))
    _log("DELETE TABLAS.dbo.UR_SF_CR18", res.rowcount)
ur_sf_cr18.to_sql("UR_SF_CR18", engine, schema="dbo", if_exists="append", index=False)
_log("ur_sf_cr18", ur_sf_cr18)


In [ ]:
# NO SE INCORPORA APERTURA PORQUE AFECTA MUCHO EL PTMO NETO DE LOS SEGUROS
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.UR_SF_CR18 SET SECTOR = 321 WHERE SECTOR = 35"))
    _log("UPDATE TABLAS.dbo.UR_SF_CR18 (sector 35->321)", res.rowcount)


In [ ]:
# IMPORTA DATA DE CCAS
t_ccast = pd.read_excel(Path("data") / "INFO_AUX" / "T_CCAST.xlsx", sheet_name="T_CCAST")
_log("t_ccast", t_ccast)


In [ ]:
# TABLAS.T_CCAST: SAS reemplazaba la tabla completa (DATA step sobre tabla ya existente)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.T_CCAST"))
    _log("DELETE TABLAS.dbo.T_CCAST", res.rowcount)
t_ccast.to_sql("T_CCAST", engine, schema="dbo", if_exists="append", index=False)
_log("t_ccast", t_ccast)


In [ ]:
# ELIMINA DE AJUSTE BONOS AÑO DE COYUNTURA PARA RECALCULAR DENUEVO POR CAMBIO DE CUENTAS INDIVIDUALES
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.AJUSTE_BONOS WHERE AÑO >= :anio"), {"anio": ANIO})
    _log("DELETE TABLAS.dbo.AJUSTE_BONOS (año coyuntura)", res.rowcount)


In [ ]:
# IMPORTA AJUSTES VARIOS DEP Y ACCIONES
aj_varios = pd.read_excel(Path("data") / "INFO_AUX" / "aj_cnsi.xlsx", sheet_name="AJUSTES_VARIOS")
# TABLAS.AJ_VARIOS: SAS reemplazaba la tabla completa (PROC IMPORT replace); se combina más abajo con AJ_D443_CR18 antes de escribir
_log("aj_varios", aj_varios)


In [ ]:
# CIERRE 2021: IMPORTA AJUSTES TRANSFERENCIAS CORRIENTES DE EMPRESAS POR CDR18
aj_d443_cr18 = pd.read_excel(Path("data") / "INFO_AUX" / "aj_cnsi.xlsx", sheet_name="base_aj_d443_cr18")
# missing numérico de SAS (año=.) es NULL en el DataFrame
aj_d443_cr18 = aj_d443_cr18[aj_d443_cr18["año"].notna()]
_log("aj_d443_cr18", aj_d443_cr18)


In [ ]:
aj_varios_final = pd.concat([aj_varios, aj_d443_cr18], ignore_index=True)
# TABLAS.AJ_VARIOS: SAS reemplazaba la tabla completa (DATA step sobre tabla ya existente)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.AJ_VARIOS"))
    _log("DELETE TABLAS.dbo.AJ_VARIOS", res.rowcount)
aj_varios_final.to_sql("AJ_VARIOS", engine, schema="dbo", if_exists="append", index=False)
_log("aj_varios_final", aj_varios_final)


In [ ]:
# AJ_d443_CR18 era una tabla WORK intermedia; en Python queda liberada al terminar el nodo, no hay tabla física que dropear


In [ ]:
# CIERRE 2021: INCORPORA AJUSTE A FBCF SECTOR FINANCIERO
fbcf_sf = pd.read_excel(Path("data") / "INFO_AUX" / "aj_fbcf_sf.xlsx", sheet_name="BASE")
# TABLAS.FBCF_SF: SAS reemplazaba la tabla completa (PROC IMPORT replace)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.FBCF_SF"))
    _log("DELETE TABLAS.dbo.FBCF_SF", res.rowcount)
fbcf_sf.to_sql("FBCF_SF", engine, schema="dbo", if_exists="append", index=False)
_log("fbcf_sf", fbcf_sf)


In [ ]:
# LIMPIA TABLAS TEMPORALES DEL FLUJO DE PROCESO — datasets WORK de la sesión SAS que no existen como tablas físicas en Python: no hay DROP TABLE que replicar, listado solo a título de trazabilidad
# WORK.SIFMI, RP_HH_SUM, RP_HH_AV, AGREGADOS, GASTO, DATA_GOB, TRANSF_K, TRANSF_K2, VAR_TK, TK_DEF, UR_PASIVO, TCAMBIO, PORC_UR_SF, ACCS_COT, ACCS_COT_37, ACCS_COT_36907, BCOS, RATIOS, VL_51021, VL_37, VL_36907, VM_BCOS, VM_EMPRESAS, T_CCAST, VM_37, VM_36907, VM_321, VL_321, VM_HOLDING, DEP_HH_FM


In [ ]:
# SE INCORPORA EN CIERRE 2022Q2. DATA BONOS EMITIDOS EN EL EXTERIOR POR NO REGULADOS DEL SECTOR FINANCIERO SECTOR=36912.
# EN CIERRE DE AÑO IMPUTAR TODA LA SERIE. CIERRE 2022: SE INCORPORA AJUSTE PARA TODA LA SERIE PARA SER CONSISTENTES
# CIERRE 2025Q2: INCORPORA EL EMISOR 33
raise NotImplementedError("Fuente 'BASE_DEUDA_EMV.sas7bdat' referenciada por ruta libre de servidor SAS ('/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/HSS/BASE_DEUDA_EMV.sas7bdat'); no es un libref de db_connections.yaml ni un archivo declarado en file_imports del nodo. Falta definir de dónde debe leerse este dataset en el nuevo entorno (¿tabla de BD? ¿archivo migrado a data/?) antes de traducir BONOS_RF_EXT, BONOS_RF_EXT_BI, RP_AUXFIN y el ajuste final a TABLAS.AJ_VARIOS.")


In [ ]:
work_conn.execute(text("""
    UPDATE #bonos_rf_ext
    SET C_SCN = 'AF.42',
        N_SCN = 'Préstamos a largo plazo',
        C_CAGENTE = '321',
        DATO = DATO * -1
"""))


In [ ]:
# acumula BONOS_RF_EXT (ya actualizado) sobre TABLAS.AJ_VARIOS
sql_append_aj_varios_1 = """
INSERT INTO TABLAS.dbo.AJ_VARIOS (MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE)
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE
FROM #bonos_rf_ext
"""
res = work_conn.execute(text(sql_append_aj_varios_1))
_log("APPEND TABLAS.dbo.AJ_VARIOS (BONOS_RF_EXT)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51_6"))
# ELIMINA BONO DEL RM ACTIVO CON CA 36 DESDE 2022, PARA CONCILIAR BIEN CON LO IMPUTADO EN LA CI DE SECTOR 36
# LO QUE ESTA INICIALMENTE SE LLEVA A EMPRESAS
sql_af32_51_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR, '51021' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN, T1.FUENTE
INTO #af32_51_6
FROM OPENROWSET(BULK '/sasdata/BCCH/GEM_DCNI/02_CNSI/02_PRE_SINTESIS/BD_CTSI_CIERRE.sas7bdat', SINGLE_BLOB) AS T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '36' AND T1.FUENTE = 'CI')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_51_6))
af32_51_6 = pd.read_sql(text("SELECT * FROM #af32_51_6"), work_conn)
_log("af32_51_6", af32_51_6)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_36_6"))
# LO QUE ESTA INICIALMENTE SE RESTA, EXCEPTO BI DE TRIM=1
sql_af32_36_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR, T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN, T1.FUENTE
INTO #af32_36_6
FROM OPENROWSET(BULK '/sasdata/BCCH/GEM_DCNI/02_CNSI/02_PRE_SINTESIS/BD_CTSI_CIERRE.sas7bdat', SINGLE_BLOB) AS T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '36' AND T1.FUENTE = 'CI')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_36_6))
af32_36_6 = pd.read_sql(text("SELECT * FROM #af32_36_6"), work_conn)
_log("af32_36_6", af32_36_6)


In [ ]:
# acumula AF32_51_6 y AF32_36_6 sobre TABLAS.AJ_VARIOS (AF32_36_6_VOL queda comentado en el SAS original, no se traduce)
cols_aj_varios = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE"
sql_append_aj_varios_2 = f"""
INSERT INTO TABLAS.dbo.AJ_VARIOS ({cols_aj_varios})
SELECT {cols_aj_varios} FROM #af32_51_6
UNION ALL
SELECT {cols_aj_varios} FROM #af32_36_6
"""
res = work_conn.execute(text(sql_append_aj_varios_2))
_log("APPEND TABLAS.dbo.AJ_VARIOS (AF32_51_6 + AF32_36_6)", res.rowcount)


In [ ]:
for t in ["#af32_51_6", "#af32_36_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## S1_d

Calcula las transferencias de capital (D.9) desde gobierno general a empresas públicas por año/trimestre, reemplaza el año de coyuntura en la tabla histórica y anexa el nuevo cálculo

*confianza: medium · verificador: approve · SAS: PROC SQL: WORK temporal desde tabla de BD + LEFT JOIN de 8 columnas con GROUP BY + DELETE/INSERT idempotente por año + PROC APPEND FORCE*

In [ ]:
# ========= S1_d =========
# obtiene transferencias de capital a empresas para imputar en la síntesis
# año debe ser mayor o igual a 2005 en cierre de año y el corriente en coyuntura
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr"))
# año interpolado como entero (no :param) para que la #tmp sobreviva
sql_ejec_cgr = f"""
SELECT *
INTO #ejec_cgr
FROM GOBGENER.dbo.EJECUCION
WHERE AÑO >= {int(ANIO)}
"""
work_conn.execute(text(sql_ejec_cgr))


In [ ]:
# transferencias de capital (D.9) a empresas: mapeo de partidas contables a SCN mediante LEFT JOIN sobre 8 columnas,
# mes convertido a trimestre con CASE WHEN, DEVENG convertido de pesos a millones
work_conn.execute(text("DROP TABLE IF EXISTS #tk_ejec"))
sql_tk_ejec = """
SELECT	  t1.AÑO,
		 (CASE WHEN t1.MES IN (1,2,3) THEN 1
			WHEN t1.MES IN (4,5,6) THEN 2
			WHEN t1.MES IN (7,8,9) THEN 3
			ELSE 4 END) AS TRIM,
			5101 AS C_SI,
			'D.9' AS C_INSTRUMENTO_SCN,
			'Capital' AS C_CUENTA,
			'H' AS C_ENTRADA,
       		 SUM(t1.DEVENG / 1000000.0) AS Dato
INTO #tk_ejec
FROM 	#ejec_cgr t1
LEFT JOIN 	GOBGENER.dbo.CR18_T_SCN_2 t2 ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
           AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
           AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO)
WHERE t1.C_CUENTA IN ('05','13','24','33') AND (t2.OBS IS NULL OR t2.OBS NOT IN ('CR18_difcoy','CR18_difact')) AND t1.C_ENTIDAD NOT IN (5601,10201) AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('TC'))
		AND t1.MONEDA = 'P' AND t2.C_SCN IN ('D91','D92','D93','D99') AND t1.C_TIPO_CUENTA = 'G' AND t2.C_CAGENTE IN ('S11','S11_EPU','tkemppúb')
		AND t2.N_CUENTA = 'capital'
GROUP BY t1.AÑO,
		 (CASE WHEN t1.MES IN (1,2,3) THEN 1
			WHEN t1.MES IN (4,5,6) THEN 2
			WHEN t1.MES IN (7,8,9) THEN 3
			ELSE 4 END)
"""
work_conn.execute(text(sql_tk_ejec))
tk_ejec = pd.read_sql(text("SELECT * FROM #tk_ejec"), work_conn)
_log("tk_ejec", tk_ejec)


In [ ]:
# elimina datos de año de coyuntura en tabla principal y en TK_EJEC (deja solo lo anterior al año de coyuntura en la tabla principal antes del anexo)
res = work_conn.execute(text("DELETE FROM TABLAS.dbo.T_TK_GG_EPU WHERE AÑO >= :anio"), {"anio": ANIO})
_log("DELETE TABLAS.dbo.T_TK_GG_EPU", res.rowcount)
res2 = work_conn.execute(text("DELETE FROM #tk_ejec WHERE AÑO < :anio"), {"anio": ANIO})
_log("DELETE #tk_ejec", res2.rowcount)


In [ ]:
# anexa TK de coyuntura a tabla principal (server-side): PROC APPEND FORCE alinea por nombre
cols_tk_gg_epu = "AÑO, TRIM, C_SI, C_INSTRUMENTO_SCN, C_CUENTA, C_ENTRADA, Dato"
sql_append_tk_gg_epu = f"""
INSERT INTO TABLAS.dbo.T_TK_GG_EPU ({cols_tk_gg_epu})
SELECT {cols_tk_gg_epu}
FROM #tk_ejec
"""
res3 = work_conn.execute(text(sql_append_tk_gg_epu))
_log("APPEND TABLAS.dbo.T_TK_GG_EPU", res3.rowcount)


In [ ]:
# limpieza de temporales de sesión (equivalente al PROC SQL DELETE EJEC_CGR, TK_EJEC)
for t in ["#ejec_cgr", "#tk_ejec"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## S1_b

Calcula el porcentaje de depósitos (DPC/DPL) destinados a hogares en fondos mutuos, imputa trimestres/años faltantes (2003-2006) y actualiza la serie histórica TABLAS.DEP_HH_FM del año de proceso en adelante

*confianza: low · verificador: revise · SAS: PROC IMPORT + múltiples PROC SQL CREATE TABLE con agregaciones/joins + generación manual de series imputadas + PROC APPEND idempotente*

In [ ]:
# ========= S1_b =========
# CALCULA % DE DEPÓSITOS PARA HOGARES EN FFMM
# Fuente: ruta absoluta del servidor SAS -> no existe en el workspace; se declara el hueco
raise NotImplementedError("ID_FM requiere el archivo SAS7BDAT '/sasdata/BCCH/GEM_DCNI/02_CNSI/12_SI_FI/33901_FFMM/IDENTIFICA_FFMM.sas7bdat' (IDENTIFICA_FFMM); no está en input_datasets ni en file_imports del nodo, no se puede leer con pyreadstat sin la ruta/mapeo real en el workspace")


In [ ]:
# 1. CALCULA % A HOGARES POR ROL DEL FONDO
# PROC IMPORT: base_datos desde Excel (hoja base_datos, datarow=2 -> header en fila 1, columnas ya con nombres)
ruta_base_datos = Path("data") / "33901_FFMM" / "BD_Patrimonio.xlsx"
base_datos = pd.read_excel(ruta_base_datos, sheet_name="base_datos", header=0)
_log("base_datos", base_datos)


In [ ]:
# PROC SQL; UPDATE base_datos SET MES=12 WHERE AÑO<2017 AND MES=.;
mask_mes_faltante = (base_datos["AÑO"] < 2017) & (base_datos["MES"].isna())
base_datos.loc[mask_mes_faltante, "MES"] = 12
_log("base_datos tras imputar MES", base_datos)


In [ ]:
# CREATE TABLE DATO_HH AS /*DATO A HOGARES*/
mask_hh = base_datos["DESTINO"].isin(["HOGARES", "EMPRE_HOGAR"])
base_hh = base_datos[mask_hh].copy()
base_hh["RUN_SDV"] = base_hh["RUN"].astype(str).str[0:4]
base_hh["_dato_calc"] = np.where(base_hh["DESTINO"] == "EMPRE_HOGAR", base_hh["PATRI_T"] / 2, base_hh["PATRI_T"])
dato_hh = (
    base_hh.groupby(["AÑO", "MES", "RUN", "tipo_fondo"], as_index=False)
    .agg(DATO=("_dato_calc", "sum"))
)
# RUN_SDV se pierde en el agrupamiento SAS por no estar en el GROUP BY; se recupera tomando el primero por combinación
dato_hh = dato_hh.merge(
    base_hh[["AÑO", "MES", "RUN", "tipo_fondo", "RUN_SDV"]].drop_duplicates(subset=["AÑO", "MES", "RUN", "tipo_fondo"]),
    on=["AÑO", "MES", "RUN", "tipo_fondo"],
    how="left",
)
_log("dato_hh", dato_hh)


In [ ]:
# CREATE TABLE DATO_TOT AS /*DATO TOTAL*/
dato_tot = (
    base_datos.groupby(["AÑO", "MES", "RUN"], as_index=False)
    .agg(DATO=("PATRI_T", "sum"))
)
dato_tot["RUN_SDV"] = base_datos.groupby(["AÑO", "MES", "RUN"])["RUN"].first().astype(str).str[0:4].values
_log("dato_tot", dato_tot)


In [ ]:
# CREATE TABLE DATO_EST AS
dato_est = dato_hh.merge(
    dato_tot, on=["AÑO", "MES", "RUN"], how="inner", suffixes=("_hh", "_tot")
)
dato_est["run_sdv"] = pd.to_numeric(dato_est["RUN_SDV_hh"], errors="coerce")
dato_est["EST"] = dato_est["DATO_hh"] / dato_est["DATO_tot"]
dato_est = dato_est[["AÑO", "MES", "RUN", "run_sdv", "tipo_fondo", "EST"]]
dato_est.columns = ["año", "mes", "run", "run_sdv", "tipo_fondo", "est"]
_log("dato_est", dato_est)


In [ ]:
# PROC SQL; DROP TABLE base_datos,DATO_HH,DATO_TOT; -- limpieza de WORK intermedios en pandas (no aplica DROP real)
del base_datos, dato_hh, dato_tot


In [ ]:
# 2. OBTIENE DATOS A APLICAR PORCENTAJE
# Fuentes SAS7BDAT en ruta absoluta del servidor -> no están disponibles en el workspace
raise NotImplementedError("data_fm_NAC requiere '/sasdata/BCCH/GEM_DCNI/02_CNSI/12_SI_FI/33901_FFMM/CARTERA_INV_NACIONAL.sas7bdat'; no está declarado en input_datasets ni file_imports del nodo")


In [ ]:
raise NotImplementedError("data_fm_ext requiere '/sasdata/BCCH/GEM_DCNI/02_CNSI/12_SI_FI/33901_FFMM/CARTERA_INV_EXTERNA.sas7bdat'; no está declarado en input_datasets ni file_imports del nodo")


In [ ]:
# DATA data_fm; SET data_fm_NAC data_fm_ext (DROP=VALOR_REL_VAL); RUN;
data_fm_ext_sin_valor_rel = data_fm_ext.drop(columns=["VALOR_REL_VAL"])
data_fm = pd.concat([data_fm_nac, data_fm_ext_sin_valor_rel], ignore_index=True)
_log("data_fm", data_fm)


In [ ]:
# PROC SQL; DROP TABLE data_fm_NAC, data_fm_ext;
del data_fm_nac, data_fm_ext_sin_valor_rel, data_fm_ext


In [ ]:
# create table data_dep_run: cruce cartera con sector del fondo (id_fm)
mask_dep = data_fm["t_instcorto"].isin(["DPC", "DPL"]) & data_fm["mes"].isin([3, 6, 9, 12])
data_fm_dep = data_fm[mask_dep]
data_dep_run = (
    data_fm_dep.merge(id_fm, on="run_fondo", how="left")
    .groupby(["año", "mes", "run_fondo", "Sector"], as_index=False)
    .agg(dato=("valor_mercado", "sum"))
)
_log("data_dep_run", data_dep_run)


In [ ]:
# create table data_dep: trimestre = mes/3 (SAS conserva decimal si mes no es múltiplo de 3; acá mes ya está filtrado a 3,6,9,12)
data_dep = (
    data_dep_run.assign(trim=data_dep_run["mes"] / 3)
    .groupby(["año", "sector" if "sector" in data_dep_run.columns else "Sector"], as_index=False)
)


In [ ]:
raise NotImplementedError("El GROUP BY de data_dep en SAS es (año, mes, sector) pero el SELECT expone trim=mes/3 sin trim en el GROUP BY; la agregación real de SAS colapsa por mes (no por trim) — se requiere confirmar con el analista si el agrupamiento pretendido es por trim o por mes antes de fijar la lógica en pandas para evitar una cifra incorrecta")


In [ ]:
# 3. CALCULA DEPOSITOS A HOGARES (año >= 2017): cruce por run_sdv=run_fondo
mask_2017 = dato_est["año"] >= 2017
dep_hh_2017 = dato_est[mask_2017].merge(
    data_dep_run,
    left_on=["año", "mes", "run_sdv"],
    right_on=["año", "mes", "run_fondo"],
    how="inner",
)
dep_hh_2017["trim"] = dep_hh_2017["mes"] / 3
dep_hh_2017["_prod"] = dep_hh_2017["est"] * dep_hh_2017["dato"]
dep_hh = (
    dep_hh_2017.groupby(["año", "mes", "Sector"], as_index=False)
    .agg(dato=("_prod", "sum"))
)
dep_hh["trim"] = dep_hh["mes"] / 3
dep_hh = dep_hh[["año", "trim", "Sector", "dato"]]
_log("dep_hh (>=2017)", dep_hh)


In [ ]:
# create table DEP_HH_2 (año < 2017): cruce sin condición de mes
mask_pre2017 = dato_est.merge(
    data_dep_run[["año", "run_fondo"]].drop_duplicates(), left_on="año", right_on="año", how="inner"
)
dep_hh2_join = dato_est.merge(
    data_dep_run,
    left_on=["año", "run_sdv"],
    right_on=["año", "run_fondo"],
    how="inner",
)
dep_hh2_join = dep_hh2_join[dep_hh2_join["año"] < 2017]
dep_hh2_join["_prod"] = dep_hh2_join["est"] * dep_hh2_join["dato"]
dep_hh_2 = (
    dep_hh2_join.groupby(["año", "mes", "Sector"], as_index=False)
    .agg(dato=("_prod", "sum"))
)
dep_hh_2["trim"] = dep_hh_2["mes"] / 3
dep_hh_2 = dep_hh_2[["año", "trim", "Sector", "dato"]]
_log("dep_hh_2 (<2017)", dep_hh_2)


In [ ]:
# DATA DEP_HH; set DEP_HH DEP_HH_2; run;
dep_hh = pd.concat([dep_hh, dep_hh_2], ignore_index=True)
_log("dep_hh (unión)", dep_hh)


In [ ]:
# create table dep_hh_fm: % total para dep de hogares
dep_hh_fm = dep_hh.merge(
    data_dep, on=["año", "trim", "Sector"], how="inner", suffixes=("_hh", "_tot")
)
dep_hh_fm["dato"] = dep_hh_fm["dato_hh"] / dep_hh_fm["dato_tot"]
dep_hh_fm = dep_hh_fm[["año", "trim", "Sector", "dato"]]
_log("dep_hh_fm", dep_hh_fm)


In [ ]:
# genera años faltantes: imputa años 2003-2005 replicando el valor de 2006 trim 1
base_imputa = dep_hh_fm[(dep_hh_fm["año"] == 2006) & (dep_hh_fm["trim"] == 1)]
imputa = base_imputa.assign(año=2005)[["año", "trim", "Sector", "dato"]]
imputa_2 = base_imputa.assign(año=2004)[["año", "trim", "Sector", "dato"]]
imputa_3 = base_imputa.assign(año=2003)[["año", "trim", "Sector", "dato"]]
imputa = pd.concat([imputa, imputa_2, imputa_3], ignore_index=True)
_log("imputa (2003-2005, trim=1)", imputa)


In [ ]:
# data imputa_a; set imputa; run; / update imputa_a set trim=2
imputa_a = imputa.copy()
imputa_a["trim"] = 2


In [ ]:
# data imputa_b; set imputa; run; / update imputa_b set trim=3
imputa_b = imputa.copy()
imputa_b["trim"] = 3


In [ ]:
# data imputa_c; set imputa; run; / update imputa_c set trim=4
imputa_c = imputa.copy()
imputa_c["trim"] = 4


In [ ]:
# data dep_hh_fm; set dep_hh_fm imputa imputa_a imputa_b imputa_c; run; /*genera base serie completa*/
dep_hh_fm = pd.concat([dep_hh_fm, imputa, imputa_a, imputa_b, imputa_c], ignore_index=True)
_log("dep_hh_fm (serie completa con imputados)", dep_hh_fm)


In [ ]:
# PROC SQL; drop table id_fm,dato_est,data_fm,data_dep_run,data_dep,dep_hh,dep_hh_2,imputa,imputa_a,imputa_b,imputa_c,imputa_2,imputa_3;
del id_fm, dato_est, data_fm, data_dep_run, data_dep, dep_hh, dep_hh_2
del imputa, imputa_a, imputa_b, imputa_c, imputa_2, imputa_3


In [ ]:
# Escritura idempotente en la BD: replica DELETE (años >= anio) + APPEND de dep_hh_fm (años >= anio, filtrado en pandas antes de subir)
# Sube dep_hh_fm a una #tmp de sesión para operar el DELETE+INSERT completamente server-side
dep_hh_fm.to_sql("#tmp_dep_hh_fm", work_conn, if_exists="replace", index=False)
res_delete = work_conn.execute(text("DELETE FROM TABLAS.dbo.DEP_HH_FM WHERE año >= :anio"), {"anio": anio})
_log("DELETE TABLAS.dbo.DEP_HH_FM (año>=anio)", res_delete.rowcount)
# el SAS también hacía DELETE FROM dep_hh_fm (WORK) WHERE año<&anio; en pandas equivale a filtrar antes del append
cols_dep_hh_fm = "año, trim, Sector, dato"
sql_append_dep_hh_fm = f"""
INSERT INTO TABLAS.dbo.DEP_HH_FM ({cols_dep_hh_fm})
SELECT {cols_dep_hh_fm}
FROM #tmp_dep_hh_fm
WHERE año >= :anio
"""
res_append = work_conn.execute(text(sql_append_dep_hh_fm), {"anio": anio})
_log("APPEND TABLAS.dbo.DEP_HH_FM", res_append.rowcount)


## Bonos_Ext

Calcula el precio implícito de bonos externos (valor de mercado / valor par) por sector, agente y trimestre, agrega recompras y estimados, deriva el balance de inicio del trimestre siguiente y reemplaza la tabla TABLAS.BONOS_EXT_PRECIO completa

*confianza: medium · verificador: approve · SAS: PROC IMPORT (2 hojas Excel) + PROC SQL con UPDATE/CREATE TABLE/agregación/UNION ALL sobre tablas temporales de sesión + DATA step de reemplazo en TABLAS*

In [ ]:
# ========= Bonos_Ext =========
# DATA DE BONOS DE LA BALANZA DE PAGOS PARA CALCULAR PRECIOS IMPLÍCITOS A USAR EN LAS CUENTAS DE GOBIERNO, EMPRESAS Y HOLDINGS
# 1. IMPORTA DATA
# ruta original: /samba/BCCH/GEM_DCNI/02_CNSI/01_SINTESIS/INFO_AUX/bonos_ext_cdr18.xlsx
bonos_ext = pd.read_excel(Path("data") / "INFO_AUX" / "bonos_ext_cdr18.xlsx", sheet_name="DATA")
_log("bonos_ext", bonos_ext)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext"))
bonos_ext.to_sql("#bonos_ext", work_conn, if_exists="replace", index=False)

work_conn.execute(text("UPDATE #bonos_ext SET CNSI = :nuevo WHERE CNSI = :viejo"), {"nuevo": 51021, "viejo": 5102})
work_conn.execute(text("UPDATE #bonos_ext SET CNSI = :nuevo WHERE CNSI = :viejo"), {"nuevo": 321, "viejo": 322})
work_conn.execute(text("UPDATE #bonos_ext SET C_CAGENTE = :nuevo WHERE C_CAGENTE <> :nuevo"), {"nuevo": "6"})
# incorporado cierre 2025q2
work_conn.execute(text("UPDATE #bonos_ext SET CNSI = :nuevo WHERE CNSI = :viejo"), {"nuevo": 36, "viejo": 33})


In [ ]:
# ruta original: /samba/BCCH/GEM_DCNI/02_CNSI/01_SINTESIS/INFO_AUX/bonos_ext_cdr18.xlsx
bonos_ext_est = pd.read_excel(Path("data") / "INFO_AUX" / "bonos_ext_cdr18.xlsx", sheet_name="DATA_EST")
_log("bonos_ext_est", bonos_ext_est)

work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext_est"))
bonos_ext_est.to_sql("#bonos_ext_est", work_conn, if_exists="replace", index=False)
res = work_conn.execute(text('DELETE FROM #bonos_ext_est WHERE [año] IS NULL'))
_log("DELETE #bonos_ext_est", res.rowcount)


In [ ]:
# CALCULA PRECIOS PARA GOBIERNO, EMPRESAS Y HOLDINGS
# CIERRE 2021: INCORPORA TMB BANCOS
# CIERRE 2022Q2: INCORPORA PRECIO DE BNOS EMITIDOS EN EL EXTERIOR DE AUXILIARES (36)
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext_precio"))
# Balance Final
sql_bonos_ext_precio = """
SELECT [Año], Trim, CNSI AS Sector, C_CAGENTE, C_SCN, C_CUENTA,
       SUM(Valor_Mercado) / SUM(Valor_par) AS Precio
INTO #bonos_ext_precio
FROM #bonos_ext
WHERE fuente IN ('Mercado Externo', 'Mercado Externo (Recompras)')
  AND cnsi IN (41, 37, 5101, 51021, 5102, 321, 36)
GROUP BY [Año], Trim, CNSI, C_CAGENTE, C_SCN, C_CUENTA
"""
work_conn.execute(text(sql_bonos_ext_precio))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext_recompra"))
# Balance Final, para recompras en empresas
sql_bonos_ext_recompra = """
SELECT [Año], Trim, CNSI AS Sector, '54' AS C_CAGENTE, C_SCN, C_CUENTA,
       SUM(Valor_Mercado) / SUM(Valor_par) AS Precio
INTO #bonos_ext_recompra
FROM #bonos_ext
WHERE fuente IN ('Mercado Externo (Recompras)')
  AND cnsi IN (5101, 51021)
GROUP BY [Año], Trim, CNSI, C_CAGENTE, C_SCN, C_CUENTA
"""
work_conn.execute(text(sql_bonos_ext_recompra))
res = work_conn.execute(text("UPDATE #bonos_ext_recompra SET Precio = 1 WHERE Precio IS NULL"))
_log("UPDATE #bonos_ext_recompra Precio faltante", res.rowcount)


In [ ]:
# data Bonos_Ext_Precio; set Bonos_Ext_Recompra Bonos_Ext_Precio BONOS_EXT_EST; run;
# el SAS REASIGNA el nombre Bonos_Ext_Precio a esta unión (recompras + precio + estimados)
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext_precio_all"))
cols_union = "[Año], Trim, Sector, C_CAGENTE, C_SCN, C_CUENTA, Precio"
sql_union = f"""
SELECT {cols_union}
INTO #bonos_ext_precio_all
FROM (
    SELECT {cols_union} FROM #bonos_ext_recompra
    UNION ALL
    SELECT {cols_union} FROM #bonos_ext_precio
    UNION ALL
    SELECT {cols_union} FROM #bonos_ext_est
) u
"""
work_conn.execute(text(sql_union))
bonos_ext_precio = pd.read_sql(text("SELECT * FROM #bonos_ext_precio_all"), work_conn)
_log("bonos_ext_precio", bonos_ext_precio)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext_precio_2"))
# Balance Inicio: traslada el precio de cierre de un trimestre al inicio del siguiente
sql_bonos_ext_precio_2 = """
SELECT
    CASE WHEN Trim = 4 THEN [Año] + 1 ELSE [Año] END AS [Año],
    CASE WHEN Trim = 4 THEN 1 ELSE Trim + 1 END AS Trim,
    Sector, C_CAGENTE, C_SCN, 'Bce Inicio' AS C_CUENTA, Precio
INTO #bonos_ext_precio_2
FROM #bonos_ext_precio_all
"""
work_conn.execute(text(sql_bonos_ext_precio_2))
bonos_ext_precio_2 = pd.read_sql(text("SELECT * FROM #bonos_ext_precio_2"), work_conn)
_log("bonos_ext_precio_2", bonos_ext_precio_2)


In [ ]:
# DATA tablas.Bonos_Ext_Precio; SET Bonos_Ext_Precio_2 Bonos_Ext_Precio; RUN;
# el SAS reemplaza la tabla TABLAS.BONOS_EXT_PRECIO con la unión Balance Inicio + Balance Final/estimados
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext_precio_final"))
cols_final = "[Año], Trim, Sector, C_CAGENTE, C_SCN, C_CUENTA, Precio"
sql_final = f"""
SELECT {cols_final}
INTO #bonos_ext_precio_final
FROM (
    SELECT {cols_final} FROM #bonos_ext_precio_2
    UNION ALL
    SELECT {cols_final} FROM #bonos_ext_precio_all
) u
"""
work_conn.execute(text(sql_final))
res = work_conn.execute(text("UPDATE #bonos_ext_precio_final SET Sector = :nuevo WHERE Sector = :viejo"), {"nuevo": 36912, "viejo": 36})
_log("UPDATE #bonos_ext_precio_final Sector", res.rowcount)


In [ ]:
# reemplazo estilo SAS: DATA step sobre tabla existente = DELETE sin WHERE + INSERT (nunca DROP/replace)
res = work_conn.execute(text("DELETE FROM TABLAS.dbo.BONOS_EXT_PRECIO"))
_log("DELETE TABLAS.dbo.BONOS_EXT_PRECIO", res.rowcount)
cols_precio = "[Año], Trim, Sector, C_CAGENTE, C_SCN, C_CUENTA, Precio"
sql_insert_final = f"""
INSERT INTO TABLAS.dbo.BONOS_EXT_PRECIO ({cols_precio})
SELECT {cols_precio} FROM #bonos_ext_precio_final
"""
res = work_conn.execute(text(sql_insert_final))
_log("INSERT TABLAS.dbo.BONOS_EXT_PRECIO", res.rowcount)


In [ ]:
# proc sql; drop table Bonos_Ext_Precio, BONOS_EXT_EST, Bonos_Ext_Precio_2, bonos_ext, Bonos_Ext_Recompra;
for t in ["#bonos_ext_precio", "#bonos_ext_est", "#bonos_ext_precio_2", "#bonos_ext", "#bonos_ext_recompra"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))
